# VAE Aug with Oversampling 

Three independent outputs are trained with BCE-based loss. Validation chooses the checkpoint; test is used only at the end.

In [2]:
from pathlib import Path
import sys
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report

sys.path.append(str(Path.cwd()))
from multilabel_utils import (
    CLASS_NAMES,
    LABEL_COLUMNS,
    MultilabelDataset,
    calculate_multilabel_metrics,
    classifier_transform,
    create_resnet18,
    get_device,
    predict_multilabel,
    set_seed,
    train_classifier,
)

SEED = 42
THRESHOLD = 0.5
EPOCHS = 5
set_seed(SEED)
device = get_device()
PROJECT_ROOT = Path.cwd().parents[1]
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "multilabel"
MODEL_DIR = PROJECT_ROOT / "models" / "multilabel"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")
print("Device:", device)
print("Split sizes:", len(train_df), len(val_df), len(test_df))

Device: mps
Split sizes: 1503 302 300


In [3]:
synthetic_df = pd.read_csv(PROJECT_ROOT / "data" / "synthetic" / "multilabel_vae" / "metadata.csv")
train_df["is_synthetic"] = False
train_df["image_path"] = ""
augmented_train_df = pd.concat([train_df, synthetic_df], ignore_index=True, sort=False)
positive_frequency = augmented_train_df[LABEL_COLUMNS].mean()
inverse_frequency = 1.0 / positive_frequency

sample_weights = (
    augmented_train_df[LABEL_COLUMNS]
    .mul(inverse_frequency)
    .max(axis=1)
)

sampler = WeightedRandomSampler(
    weights=torch.tensor(
        sample_weights.to_numpy(),
        dtype=torch.double,
    ),
    num_samples=len(augmented_train_df),
    replacement=True,
    generator=torch.Generator().manual_seed(SEED),
)

print("Positive frequencies:")
print(positive_frequency)
print("Real training images:", len(train_df))
print("Synthetic training images:", len(synthetic_df))
print(augmented_train_df[LABEL_COLUMNS].sum())

transform = classifier_transform()
train_loader = DataLoader(
    MultilabelDataset(
        augmented_train_df,
        PROJECT_ROOT,
        transform,
    ),
    batch_size=32,
    sampler=sampler,
)
val_loader = DataLoader(MultilabelDataset(val_df, PROJECT_ROOT, transform), batch_size=32)
test_loader = DataLoader(MultilabelDataset(test_df, PROJECT_ROOT, transform), batch_size=32)

Positive frequencies:
Surface_Crack    0.831253
Delamination     0.222666
Pinhole          0.328507
dtype: float64
Real training images: 1503
Synthetic training images: 500
Surface_Crack    1665
Delamination      446
Pinhole           658
dtype: int64


In [4]:
model = create_resnet18(len(LABEL_COLUMNS)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
checkpoint_path = MODEL_DIR / "vae_oversampling_best.pth"

history = train_classifier(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    checkpoint_path,
    epochs=EPOCHS,
)

Epoch 1/5 | Train loss: 0.3604 | Validation loss: 0.1978
Epoch 2/5 | Train loss: 0.2280 | Validation loss: 0.1529
Epoch 3/5 | Train loss: 0.1621 | Validation loss: 0.1168
Epoch 4/5 | Train loss: 0.1263 | Validation loss: 0.1352
Epoch 5/5 | Train loss: 0.1083 | Validation loss: 0.1291


In [5]:
model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
true_labels, probabilities, predictions = predict_multilabel(
    model, test_loader, device, threshold=THRESHOLD
)

print(classification_report(
    true_labels,
    predictions,
    target_names=CLASS_NAMES,
    zero_division=0,
))

metrics = calculate_multilabel_metrics(
    true_labels,
    predictions,
    "VAE + Oversampling",
)
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(
    PROCESSED_DIR / "vae_oversampling_metrics.csv",
    index=False,
)
metrics_df.round(4)

               precision    recall  f1-score   support

Surface Crack       0.98      0.98      0.98       276
 Delamination       1.00      0.81      0.89        26
      Pinhole       0.90      0.97      0.93        73

    micro avg       0.97      0.97      0.97       375
    macro avg       0.96      0.92      0.94       375
 weighted avg       0.97      0.97      0.97       375
  samples avg       0.98      0.98      0.97       375



,Model,Exact Match Accuracy,Hamming Loss,Micro F1,Macro F1,Surface Crack F1,Delamination F1,Pinhole F1
0,VAE + Oversampling,0.9333,0.0278,0.9667,0.9366,0.9819,0.8936,0.9342


## Interpretation

Focus on macro F1 and the minority-label F1 scores. Exact-match accuracy requires the entire three-value vector to be correct.